# 6. 信号与组合：从六模型滚动预测到目标持仓

本 notebook 接在 `model.ipynb` 后面，读取已经保存的滚动预测；无需再次训练模型，也不执行其他 notebook。

**本次要完成的链路：原始预测 → 校验来源和时间 → 交易信号 → 横截面排名 → 目标权重 → 股数调整意图 → 分区保存与重新读取。**

核心函数写在本 notebook 内，重新启动内核后从上到下运行即可。每个步骤都说明输入、处理、输出和检查方法。

第一版采用日频、沪深股票、只做多和次日收盘参考成交。账户及真实成交尚未接入时，组合示例使用明确标记的空仓快照，产生的是 **idealized_preview**；真实持仓路径与净值由后续回测内核生成。

当前原始数据缺少独立的历史 ST/停复牌事件表，默认运行模式为 `research`，并保存 `research_only` 标记。`strict` 模式会阻止未知状态下的新买入。两种模式都不会借助未来收益标签判断股票资格。

## 1. 配置运行范围与组合基线

**输入**：项目路径、已完成的滚动预测运行以及事前固定的组合规则。

`ROLLING_RUN=None` 表示首次读取最新成功运行，然后把具体目录固定下来。将它改成指定 `run_...` 路径可以重现旧版本。程序不在测试收益里选择模型，六模型全部保留。

- `top_k=50`：最多规划 50 个股票席位。
- `target_stock_weight=0.95`：规划股票仓位 95%，给现金及执行误差留出空间。
- 每个新席位的起始预算为 `0.95/50=1.9%`，候选不足时保留现金。
- `max_discretionary_replacements=5`：正常缓冲轮换最多提出 5 个旧仓完整退出；首次空仓建仓不受此数误限。这是股票替换数，不是金额换手率。
- `min_cross_section=30`：可用横截面太小或分数全部相同，就暂停新增买入。
- 费用参数是**示例预算假设**，只用于检查股数意图能否付得起；成交后的实际费用由后续账户统一记账。

**输出**：`SIGNAL_CONFIG` 和独立的 `PORTFOLIO_CONFIG`。前者影响信号数据，后者可以在同一批信号上比较不同组合规则。

**检查**：这些数值是开发起始基线，修改后会形成新的配置哈希；它们不代表已经验证的最优策略。内核请选择安装了 pandas、numpy、pyarrow 的 `factor_research` 环境。

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import copy
import contextlib
import io as text_io
import json
import time
import uuid
import numpy as np
import pandas as pd
import pyarrow
from IPython.display import display

PROJECT_DIR = Path(r"E:\document-prefer\python for finanace\multi-model-bactest")
ROLLING_RUN = None
OUTPUT_ROOT = PROJECT_DIR / "data/processed/signals"
VERIFY_SOURCE_HASHES = True

SIGNAL_CONFIG = {
    "status_mode": "research",   # strict：未知历史交易状态禁止新买入
    "min_cross_section": 30,
    "signal_delay_minutes": 30,  # 日线研究假设：收盘后30分钟可用
    "timezone": "Asia/Shanghai",
    "execution_reference": "next_trading_day_close",
}
PORTFOLIO_CONFIG = {
    **SIGNAL_CONFIG,
    "policy": "topk_buffer",    # 也支持 topk_equal
    "top_k": 50,
    "max_discretionary_replacements": 5,
    "target_stock_weight": 0.95,
    "single_name_weight_cap": 0.03,
    "commission_rate_estimate": 0.0003,
    "min_commission_estimate": 5.0,
    "other_buy_fee_rate_estimate": 0.0,
}
RUN_EXPORT = True
REUSE_MATCHING_RUN = True
PREVIEW_INITIAL_CASH = 1_000_000.0  # 只用于示例空仓快照
print("pandas", pd.__version__, "pyarrow", pyarrow.__version__)
print("状态模式：", SIGNAL_CONFIG["status_mode"])
display(pd.Series(PORTFOLIO_CONFIG, name="value").to_frame())


pandas 3.1.0 pyarrow 26.0.0
状态模式： research
                                                 value
status_mode                                   research
min_cross_section                                   30
signal_delay_minutes                                30
timezone                                 Asia/Shanghai
execution_reference             next_trading_day_close
policy                                     topk_buffer
top_k                                               50
max_discretionary_replacements                       5
target_stock_weight                               0.95
single_name_weight_cap                            0.03
commission_rate_estimate                        0.0003
min_commission_estimate                            5.0
other_buy_fee_rate_estimate                        0.0


## 2. 定义文件校验与安全保存工具

**为什么需要**：同一个文件名可能对应不同版本，只有路径不足以保证复现。SHA256 对文件内容生成校验值，配置哈希则区分不同规则。

**处理**：逐块读取文件计算哈希，避免把整个模型文件放入内存；JSON 先写临时文件再替换。latest 指针只能指向对应目录的直接子运行。

**输出**：下面是通用工具函数，此格只定义函数，不读取或训练模型。后续任何一步校验失败都会报出具体文件。

In [2]:
import hashlib
import json
import uuid
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

SIGNAL_SCHEMA = "signal_dataset_v1"
EXPECTED_MODELS = ["ols", "lasso", "random_forest", "xgboost", "lightgbm", "mlp"]
FORBIDDEN_DECISION_FIELDS = {"label", "label_end_date", "label_usable", "evaluation_label", "target", "y_raw", "supervised_eligible"}


def read_json_file(path):
    return json.loads(Path(path).read_text(encoding="utf-8-sig"))


def stream_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def to_json_value(value):
    if isinstance(value, dict):
        return {str(k): to_json_value(v) for k, v in value.items()}
    if isinstance(value, (list, tuple, np.ndarray)):
        return [to_json_value(v) for v in value]
    if isinstance(value, np.generic):
        return to_json_value(value.item())
    if isinstance(value, (Path, pd.Timestamp)):
        return str(value)
    if isinstance(value, float) and not np.isfinite(value):
        return None
    return value


def stable_hash(value):
    data = json.dumps(to_json_value(value), sort_keys=True, allow_nan=False)
    return hashlib.sha256(data.encode("utf-8")).hexdigest()


def save_json_atomic(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(json.dumps(to_json_value(value), ensure_ascii=False,
                                    indent=2, allow_nan=False), encoding="utf-8")
    temporary.replace(path)


def resolve_latest(root):
    root = Path(root).resolve()
    run = (root / read_json_file(root / "latest_run.json")["run_dir"]).resolve()
    if run.parent != root:
        raise ValueError("latest 指针必须指向本目录的直接子运行。")
    return run


def require_hash(path, expected, enabled=True):
    if not Path(path).is_file():
        raise FileNotFoundError(path)
    if enabled and (not expected or stream_sha256(path) != expected):
        raise ValueError(f"文件校验失败：{path}")


## 3. 固定并核验滚动预测来源

**输入**：滚动运行的 metadata/config/windows、六个预测文件、基础训练和每折模型摘要。

**处理**：检查运行完成、六模型顺序一致、因子和数据来源一致、冻结参数一致、各折时间不重叠、训练标签在窗口开始前已经实现。再核对预测、模型文件及上游配置的内容哈希。这里只读取模型文件的字节用于校验，**不加载 pickle、不重新预测或训练**。

**输出**：`source` 保存固定目录、模型清单、窗口、行数、文件路径和哈希。

**如何检查**：表里应该有六个模型。配置中的 `train_months` 是这个实际运行使用的训练窗口，不能根据 notebook 默认值猜测。当前项目的运行是最近 8 个月训练、每 3 个月重训。

In [3]:
def load_signal_source(project_dir, rolling_run=None, verify_hashes=True):
    """固定完整滚动运行；只读 JSON/Parquet，不加载模型 pickle。"""
    project_dir = Path(project_dir).resolve()
    root = project_dir / "data/processed/model_training/rolling"
    run = Path(rolling_run).resolve() if rolling_run is not None else resolve_latest(root)
    meta, config = read_json_file(run / "metadata.json"), read_json_file(run / "config.json")
    if meta.get("status") != "complete" or meta.get("role") != "walk_forward_test":
        raise ValueError("输入必须是完整的 walk_forward_test 运行。")
    if meta["models"] != EXPECTED_MODELS or config["models"] != EXPECTED_MODELS:
        raise ValueError("本版要求固定的六个模型，名称和顺序必须一致。")
    base = Path(meta["base_run"]).resolve()
    base_meta, base_config = read_json_file(base / "metadata.json"), read_json_file(base / "config.json")
    if base_meta.get("status") != "complete" or stable_hash(base_config) != base_meta["config_sha256"]:
        raise ValueError("基础训练运行未完成或配置校验失败。")
    params = read_json_file(base / "best_params.json")
    require_hash(base / "best_params.json", base_meta["best_params_sha256"], verify_hashes)
    if stable_hash(config["best_params"]) != meta["frozen_params_sha256"] or stable_hash(params) != meta["frozen_params_sha256"]:
        raise ValueError("滚动超参数与基础训练冻结参数不一致。")
    if meta["source"] != base_meta["source"] or config["source"] != meta["source"]:
        raise ValueError("基础运行和滚动运行的因子/数据版本不一致。")
    upstream = meta["source"]
    for directory, filename, key in [
        (upstream["model_data_run"], "metadata.json", "model_metadata_sha256"),
        (upstream["model_data_run"], "config.json", "model_config_sha256"),
        (upstream["selection_run"], "selected_factors.json", "selection_sha256"),
        (upstream["selection_run"], "metadata.json", "selection_metadata_sha256"),
        (upstream["selection_run"], "source.json", "selection_source_sha256"),
        (upstream["alpha_run"], "metadata.json", "alpha_metadata_sha256"),
    ]:
        require_hash(Path(directory) / filename, upstream[key], verify_hashes)
    if read_json_file(Path(upstream["selection_run"]) / "selected_factors.json") != upstream["features"]:
        raise ValueError("因子顺序与选择运行不一致。")
    windows = pd.read_csv(run / "windows.csv", parse_dates=["predict_start", "predict_end", "max_train_label_end"])
    windows = windows.sort_values("fold").reset_index(drop=True)
    if len(windows) != meta["folds"] or windows["fold"].duplicated().any():
        raise ValueError("滚动窗口清单缺失或重复。")
    if windows[["predict_start","predict_end","max_train_label_end"]].isna().any().any():
        raise ValueError("滚动窗口的时间边界缺失。")
    previous_end, headers, paths = None, {}, {}
    for row in windows.itertuples(index=False):
        if row.predict_start > row.predict_end or (previous_end is not None and row.predict_start <= previous_end):
            raise ValueError("滚动窗口为空或发生重叠。")
        if row.max_train_label_end >= row.predict_start:
            raise ValueError("训练标签在滚动窗口开始时尚未实现。")
        previous_end = row.predict_end
        for model in EXPECTED_MODELS:
            header_path = run / f"fold-{row.fold:03d}/models/{model}.json"
            header = read_json_file(header_path)
            if (header["model"] != model or header["fold"] != row.fold
                    or header["features"] != upstream["features"] or header["source"] != upstream
                    or stable_hash(header["params"]) != stable_hash(params[model])
                    or header["frozen_params_sha256"] != meta["frozen_params_sha256"]):
                raise ValueError(f"每折模型摘要不一致：{model}, fold={row.fold}")
            if pd.isna(pd.Timestamp(header["max_train_label_end"])) or pd.isna(pd.Timestamp(header["available_after"])):
                raise ValueError("每折模型的可用日期或标签截止日期缺失。")
            if pd.Timestamp(header["max_train_label_end"]) >= row.predict_start:
                raise ValueError("每折模型包含未来监督标签。")
            require_hash(header_path.with_suffix(".pkl"), header["model_sha256"], verify_hashes)
            headers[(int(row.fold), model)] = header
    counts = {}
    for model in EXPECTED_MODELS:
        path = run / f"{model}_predictions.parquet"
        require_hash(path, meta["prediction_sha256"][model], verify_hashes)
        parquet = pq.ParquetFile(path)
        needed = {"datetime", "instrument", "score", "model", "role", "fold"}
        if not needed.issubset(parquet.schema_arrow.names):
            raise ValueError(f"预测字段缺失：{model}")
        paths[model], counts[model] = path, parquet.metadata.num_rows
    if len(set(counts.values())) != 1:
        raise ValueError("六模型上游覆盖行数不同，需要先审计原因。")
    fingerprint = dict(rolling_run=str(run), metadata_sha256=stream_sha256(run/"metadata.json"),
                       config_sha256=stream_sha256(run/"config.json"), windows_sha256=stream_sha256(run/"windows.csv"),
                       source=upstream, prediction_sha256=meta["prediction_sha256"])
    return dict(project_dir=project_dir, rolling_run=run, base_run=base, models=EXPECTED_MODELS.copy(),
                metadata=meta, config=config, windows=windows, fold_headers=headers,
                prediction_paths=paths, row_counts=counts, source_hash=stable_hash(fingerprint),
                manifest=fingerprint, verified_hashes=bool(verify_hashes))


In [4]:
source = load_signal_source(PROJECT_DIR, ROLLING_RUN, VERIFY_SOURCE_HASHES)
print("已绑定滚动运行：", source["rolling_run"])
print("基础训练运行：", source["base_run"])
print("最近训练月数：", source["config"]["train_months"],
      "；重训间隔月数：", source["config"]["retrain_months"])
display(pd.DataFrame({"model": source["models"],
                      "prediction_rows": [source["row_counts"][m] for m in source["models"]]}))
display(source["windows"][["fold", "predict_start", "predict_end", "max_train_label_end", "predict_rows"]])


已绑定滚动运行： E:\document-prefer\python for finanace\multi-model-bactest\data\processed\model_training\rolling\run_20261025T122254Z_28994c61
基础训练运行： E:\document-prefer\python for finanace\multi-model-bactest\data\processed\model_training\run_20261025T114810Z_ce49dd93
最近训练月数： 8 ；重训间隔月数： 3


,model,prediction_rows
0,ols,1242601
1,lasso,1242601
2,random_forest,1242601
3,xgboost,1242601
4,lightgbm,1242601
5,mlp,1242601


,fold,predict_start,predict_end,max_train_label_end,predict_rows
0,0,2025-01-01,2025-03-31,2024-12-31,290388
1,1,2025-04-01,2025-06-30,2025-03-31,305979
2,2,2025-07-01,2025-09-30,2025-06-30,338180
3,3,2025-10-01,2025-12-31,2025-09-30,308054


## 4. 准备信号日行情与完整市场交易日历

**输入**：项目已有的交易日历、股票上市日期和各交易日原始日线。

**处理**：保留未复权价格；成交量从“手”转为“股”，成交额从“千元”转为“元”。按日检查日期和重复代码。只将某一天的行情连接到同一天的信号，不用下一天的最终价格筛选今天的股票。

当前基础表的名称、上市状态和行业不构成完整历史 ST/行业记录，因此程序只使用其中的上市日期。`historical_status_verified=False` 明确保留这个数据缺口：research 模式使用带标记的研究假设，strict 模式不给未知状态新买入资格。

**输出**：`market['daily']`、`market['calendar']`、`market['listing']` 及原始文件哈希。

**如何检查**：交易日历最后一天目前是 2025-12-31。它决定末端信号是否有可映射的下一交易日；没有下一日时保留信号并标记等待日历。

In [5]:
def load_known_market(source):
    """日线只用于各自的信号日；不以当前名称/行业冒充历史 ST/行业。"""
    raw_root = source["project_dir"] / "data/raw"
    cal_paths = sorted((raw_root/"basic").glob("trade_dates_*.csv"))
    if not cal_paths:
        raise FileNotFoundError("缺少 trade_dates 交易日历。")
    calendars = []
    for path in cal_paths:
        frame = pd.read_csv(path, dtype=str, encoding="utf-8-sig")
        calendars.append(pd.to_datetime(frame["trade_date"], format="%Y%m%d"))
    calendar = pd.DatetimeIndex(pd.concat(calendars).drop_duplicates().sort_values())
    if calendar.empty:
        raise ValueError("交易日历为空。")
    basic_path = raw_root/"basic/stock_basic.csv"
    listing = pd.read_csv(basic_path, dtype=str, encoding="utf-8-sig")
    listing["list_date"] = pd.to_datetime(listing["list_date"], format="%Y%m%d", errors="coerce")
    if listing["ts_code"].duplicated().any():
        raise ValueError("股票基础表代码重复。")
    listing = listing.set_index("ts_code")[["list_date"]]
    listing.index.name = "instrument"
    start, end = pd.Timestamp(source["metadata"]["start"]), pd.Timestamp(source["metadata"]["end"])
    days = calendar[(calendar >= start) & (calendar <= end)]
    pieces, manifest = [], []
    columns = ["ts_code", "trade_date", "open", "high", "low", "close", "pre_close", "vol", "amount"]
    for day in days:
        path = raw_root/"daily"/str(day.year)/f"{day:%Y%m%d}.csv"
        if not path.exists():
            raise FileNotFoundError(f"交易日日线文件缺失：{path}")
        frame = pd.read_csv(path, usecols=columns, dtype={"ts_code":str,"trade_date":str})
        if not frame["trade_date"].eq(day.strftime("%Y%m%d")).all() or frame.ts_code.duplicated().any():
            raise ValueError(f"日线日期或唯一性错误：{path.name}")
        frame = frame.rename(columns={"ts_code":"instrument"}).drop(columns="trade_date")
        frame["signal_date"] = day
        for field in ["open", "high", "low", "close", "pre_close", "vol", "amount"]:
            frame[field] = pd.to_numeric(frame[field], errors="coerce")
        frame["volume_shares"], frame["amount_yuan"] = frame.pop("vol")*100, frame.pop("amount")*1000
        pieces.append(frame)
        manifest.append(dict(path=str(path), sha256=stream_sha256(path), rows=len(frame)))
    daily = pd.concat(pieces, ignore_index=True).set_index(["signal_date","instrument"]).sort_index()
    daily["market_observed"] = (np.isfinite(daily["close"]) & daily["close"].gt(0)
                                  & daily["volume_shares"].gt(0) & daily["amount_yuan"].ge(0))
    daily["historical_status_verified"] = False
    daily["known_can_buy"], daily["known_can_sell"] = False, False
    return dict(daily=daily, calendar=calendar, listing=listing,
                source_files=manifest+[dict(path=str(basic_path),sha256=stream_sha256(basic_path))]
                +[dict(path=str(p),sha256=stream_sha256(p)) for p in cal_paths],
                assumptions=["raw_daily_only", "historical_ST_and_suspension_state_unavailable",
                             "no_current_name_or_industry_used_as_history"])


In [6]:
market = load_known_market(source)
print(f"读取原始行情：{len(market['daily']):,} 行")
print("交易日历范围：", market["calendar"].min().date(), "~", market["calendar"].max().date())
print("研究假设：", market["assumptions"])
display(market["daily"].head(5))


读取原始行情：1,313,926 行
交易日历范围： 2022-01-04 ~ 2025-12-31
研究假设： ['raw_daily_only', 'historical_ST_and_suspension_state_unavailable', 'no_current_name_or_industry_used_as_history']


open   high  ...  known_can_buy  known_can_sell
signal_date instrument                ...                               
2025-01-02  000001.SZ   11.73  11.77  ...          False           False
            000002.SZ    7.25   7.36  ...          False           False
            000004.SZ   13.73  14.39  ...          False           False
            000006.SZ    7.23   7.52  ...          False           False
            000007.SZ    7.00   7.27  ...          False           False

[5 rows x 11 columns]

## 5. 将预测标准化成交易信号，再计算排名

**输入**：一个模型的原始预测、固定来源、信号日行情和配置。

**处理分两层**：

1. 标准化：通过列允许清单只读取 `score/model/role/fold` 和索引。未来 `label`、`label_usable`、`evaluation_label` 等列不会进入组合输入。关联每折模型可用日期，并用交易日历求真正的下一交易日。
2. 排名：同日同模型按分数降序、股票代码升序排列。分位数使用平均并列名次，同分不会因代码先后产生虚假的预测差距。

`candidate_eligible` 表示单只股票的新买入资格；`cross_section_informative` 表示整个横截面是否足够大且有不同分数；`actionable` 同时满足两者。Lasso 若收缩为常数预测，程序会显示分数无区分度并暂停新增买入，不能用代码排序冒充有效选股。

**时间解释**：`signal_date=t`；`signal_available_at` 是 t 日收盘后登记的研究处理时点；`earliest_execution_date` 是下一个交易日。真实价格、成交率和资金检查留给执行层。

**输出**：统一信号表和排名。即使缺未来收益标签，预测行仍保留；只有不满足交易条件时才失去新增买入资格。

In [7]:
def normalize_model_signals(source, model, market, config):
    """通过列允许清单读取，不将未来标签带入交易接口。"""
    if model not in source["models"]:
        raise ValueError("模型不在冻结清单。")
    mode = config.get("status_mode", "research")
    if mode not in {"research","strict"}:
        raise ValueError("status_mode 必须为 research 或 strict。")
    frame = pd.read_parquet(source["prediction_paths"][model], columns=["score","model","role","fold"]).reset_index()
    frame = frame.rename(columns={"datetime":"signal_date","score":"score_raw"})
    frame["signal_date"] = pd.to_datetime(frame["signal_date"])
    if len(frame) != source["row_counts"][model] or frame.duplicated(["signal_date","instrument"]).any():
        raise ValueError("预测行数错误或主键重复。")
    if not frame["model"].eq(model).all() or not frame["role"].eq("walk_forward_test").all():
        raise ValueError("预测模型或角色错误。")
    frame = frame.rename(columns={"role":"prediction_role"})
    frame["model_available_after"] = pd.NaT
    frame["max_train_label_end"] = pd.NaT
    for window in source["windows"].itertuples(index=False):
        selected = frame["fold"].eq(window.fold)
        header = source["fold_headers"][(int(window.fold),model)]
        if int(selected.sum()) != int(window.predict_rows):
            raise ValueError("每折预测行数与清单不一致。")
        days = frame.loc[selected,"signal_date"]
        if not days.between(window.predict_start,window.predict_end).all():
            raise ValueError("预测日期超出对应窗口。")
        if not days.gt(pd.Timestamp(header["available_after"])).all():
            raise ValueError("模型被用于可用日期之前。")
        frame.loc[selected,"model_available_after"] = pd.Timestamp(header["available_after"])
        frame.loc[selected,"max_train_label_end"] = pd.Timestamp(header["max_train_label_end"])
    if frame["model_available_after"].isna().any():
        raise ValueError("未知的滚动窗口。")
    calendar = market["calendar"]
    lookup = pd.Series(calendar[1:].to_numpy(), index=calendar[:-1])
    if not frame["signal_date"].isin(calendar).all():
        raise ValueError("信号日期不在市场交易日历。")
    frame["earliest_execution_date"] = frame["signal_date"].map(lookup)
    zone = config.get("timezone","Asia/Shanghai")
    frame["signal_available_at"] = (frame["signal_date"]+pd.Timedelta(hours=15,
        minutes=int(config.get("signal_delay_minutes",30)))).dt.tz_localize(zone)
    frame["earliest_execution_at"] = (frame["earliest_execution_date"]+pd.Timedelta(hours=15)).dt.tz_localize(zone)
    observed = market["daily"][["close","pre_close","volume_shares","amount_yuan","market_observed",
                                  "historical_status_verified","known_can_buy","known_can_sell"]].reset_index()
    frame = frame.merge(observed, on=["signal_date","instrument"], how="left", validate="one_to_one")
    frame = frame.merge(market["listing"].reset_index(),on="instrument",how="left",validate="many_to_one")
    finite = np.isfinite(frame["score_raw"])
    listed = frame["list_date"].notna() & frame["list_date"].le(frame["signal_date"])
    liquid = frame["market_observed"].fillna(False).astype(bool)
    status_known = frame["historical_status_verified"].fillna(False).astype(bool)
    status_ok = frame["known_can_buy"].fillna(False).astype(bool)
    if mode == "research":
        status_ok = status_ok | ~status_known
    future_day = frame["earliest_execution_date"].notna()
    frame["candidate_eligible"] = finite & listed & liquid & status_ok & future_day
    frame["reason_code"] = np.select([~finite,~listed,~liquid,~future_day,~status_ok,~status_known],
        ["invalid_score","listing_history_unknown_or_prelisting","no_observed_liquid_market",
         "awaiting_calendar","historical_status_unknown_or_restricted","assumed_historical_status"],default="eligible")
    frame["eligibility_status"] = np.where(frame["candidate_eligible"],"eligible", "excluded")
    frame["result_level"] = "research_only" if mode == "research" else "strict_status_check"
    frame["rolling_run"], frame["base_model_run"] = source["rolling_run"].name, source["base_run"].name
    frame["source_hash"], frame["config_hash"] = source["source_hash"], stable_hash(config)
    if FORBIDDEN_DECISION_FIELDS.intersection(frame.columns):
        raise AssertionError("未来监督信息进入交易信号。")
    return frame.sort_values(["signal_date","instrument"]).reset_index(drop=True)


def rank_signals(signals, config):
    if FORBIDDEN_DECISION_FIELDS.intersection(signals.columns):
        raise ValueError("rank_signals 不接受标签列。")
    frame = signals.copy()
    keys = ["signal_date","model"]
    valid = frame["candidate_eligible"].eq(True) & np.isfinite(frame["score_raw"])
    chosen = frame.loc[valid].sort_values(keys+["score_raw","instrument"],ascending=[True,True,False,True])
    frame["score_rank"], frame["score_percentile"] = np.nan, np.nan
    if len(chosen):
        grouped = chosen.groupby(keys,sort=False)
        chosen_rank = grouped.cumcount()+1
        average_rank = grouped["score_raw"].rank(ascending=False,method="average")
        n = grouped["score_raw"].transform("size")
        frame.loc[chosen.index,"score_rank"] = chosen_rank.to_numpy(dtype=float)
        frame.loc[chosen.index,"score_percentile"] = ((n-average_rank)/(n-1).replace(0,np.nan)).to_numpy()
    counts = frame.assign(_valid=valid, _value=frame["score_raw"].where(valid)).groupby(keys).agg(
        candidate_count=("_valid","sum"), distinct_scores=("_value","nunique"))
    frame = frame.join(counts,on=keys)
    frame["cross_section_informative"] = (frame["candidate_count"].ge(config.get("min_cross_section",30))
                                            & frame["distinct_scores"].gt(1))
    frame["actionable"] = frame["candidate_eligible"] & frame["cross_section_informative"]
    frame["cross_section_status"] = np.select([frame["candidate_count"].lt(config.get("min_cross_section",30)),
        frame["distinct_scores"].le(1)], ["insufficient_coverage","uninformative_cross_section"],default="ok")
    return frame.sort_values(keys+["score_rank","instrument"],na_position="last").reset_index(drop=True)


In [8]:
demo_model = "ols"  # 只作接口示例，不是按测试收益选出的模型
demo_ranked = rank_signals(normalize_model_signals(source, demo_model, market, SIGNAL_CONFIG), SIGNAL_CONFIG)
PREVIEW_DAY = demo_ranked["signal_date"].min()
demo_day = demo_ranked.loc[demo_ranked.signal_date.eq(PREVIEW_DAY)].copy()
print("示例信号日：", PREVIEW_DAY.date(), "；最早执行日：", demo_day.earliest_execution_date.iloc[0])
display(demo_day[["signal_date", "instrument", "score_raw", "score_rank", "score_percentile",
                  "candidate_eligible", "cross_section_informative", "reason_code"]].head(10))
assert not FORBIDDEN_DECISION_FIELDS.intersection(demo_ranked.columns)
assert len(demo_ranked) == source["row_counts"][demo_model]
print("通过：上游行数全部保留，未来标签没有进入交易表。")
del demo_ranked


示例信号日： 2025-01-02 ；最早执行日： 2025-01-03 00:00:00


,signal_date,instrument,score_raw,score_rank,score_percentile,candidate_eligible,cross_section_informative,reason_code
0,2025-01-02,301165.SZ,0.023968,1.0,1.000000,True,True,assumed_historical_status
1,2025-01-02,301110.SZ,0.021215,2.0,0.999803,True,True,assumed_historical_status
2,2025-01-02,300125.SZ,0.020528,3.0,0.999606,True,True,assumed_historical_status
3,2025-01-02,300007.SZ,0.020269,4.0,0.999410,True,True,assumed_historical_status
4,2025-01-02,300399.SZ,0.019601,5.0,0.999213,True,True,assumed_historical_status
5,2025-01-02,603955.SH,0.019238,6.0,0.999016,True,True,assumed_historical_status
6,2025-01-02,300499.SZ,0.019189,7.0,0.998819,True,True,assumed_historical_status
7,2025-01-02,002329.SZ,0.019100,8.0,0.998622,True,True,assumed_historical_status
8,2025-01-02,000833.SZ,0.018844,9.0,0.998426,True,True,assumed_historical_status
9,2025-01-02,300276.SZ,0.018702,10.0,0.998229,True,True,assumed_historical_status


通过：上游行数全部保留，未来标签没有进入交易表。


## 6. 分模型、分月份保存，并提供统一读取接口

**为什么分区**：六模型合计约 745 万行。逐模型处理、逐月份写 Parquet，能控制内存，也方便后面只读一个日期窗口。

**处理**：保存含资格与排名的完整 `signals`，另保存较窄的 `rankings`；生成每日覆盖、常数分数、排除原因和日历缺口报告。只有六模型全部成功才更新 latest；失败目录保留失败状态用于排查，旧成功指针保持不变。

**输出**：`data/processed/signals/run_.../`。相同来源、行情文件及配置的成功版本可以校验后复用。

读取接口：`load_signal_data(..., model, start, end)` 适合单日/短窗口；`iter_signal_batches(...)` 适合后续引擎分月读取。日期和模型筛选不依赖未来标签。

In [9]:
def export_signal_dataset(source, market, config, output_root):
    """逐模型、逐月份导出，失败不更新最新成功指针。"""
    root = Path(output_root)
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    run = root/f"run_{stamp}_{uuid.uuid4().hex[:8]}"
    run.mkdir(parents=True,exist_ok=False)
    (run/"reports").mkdir()
    save_json_atomic(run/"config.json",config)
    save_json_atomic(run/"source.json",dict(prediction=source["manifest"],market_files=market["source_files"],
        verified_hashes=source["verified_hashes"],assumptions=market["assumptions"]))
    metadata = dict(schema=SIGNAL_SCHEMA,status="running",models=source["models"],
        rolling_run=str(source["rolling_run"]),source_hash=source["source_hash"],config_hash=stable_hash(config),
        result_level="research_only" if config["status_mode"]=="research" else "strict_status_check")
    save_json_atomic(run/"metadata.json",metadata)
    counts, inventory, coverage, exclusions = {}, {}, [], []
    try:
        for model in source["models"]:
            print(f"处理 {model}：读取、校验、排名并保存……",flush=True)
            frame = rank_signals(normalize_model_signals(source,model,market,config),config)
            counts[model] = len(frame)
            report = frame.groupby(["signal_date","model"],as_index=False).agg(
                rows=("instrument","size"), candidates=("candidate_eligible","sum"),
                actionable=("actionable","sum"), informative=("cross_section_informative","first"),
                score_min=("score_raw","min"),score_max=("score_raw","max"),score_std=("score_raw","std"),
                calendar_available=("earliest_execution_date",lambda s:s.notna().all()))
            coverage.append(report)
            exclusions.append(frame.groupby(["model","reason_code"]).size().rename("rows").reset_index())
            folder = run/"signals"/f"model={model}"
            ranks = run/"rankings"/f"model={model}"
            folder.mkdir(parents=True); ranks.mkdir(parents=True)
            for month, block in frame.groupby(frame["signal_date"].dt.strftime("%Y%m"),sort=True):
                path = folder/f"part-{month}.parquet"
                block.to_parquet(path,index=False,compression="snappy")
                inventory[str(path.relative_to(run))] = stream_sha256(path)
                ranking = block[["signal_date","instrument","model","score_rank","score_percentile","actionable"]]
                path = ranks/f"part-{month}.parquet"
                ranking.to_parquet(path,index=False,compression="snappy")
                inventory[str(path.relative_to(run))] = stream_sha256(path)
            print(f"  {len(frame):,} 行；可发起新买入的信号 {int(frame.actionable.sum()):,} 行。",flush=True)
            del frame
        daily = pd.concat(coverage,ignore_index=True)
        daily.to_csv(run/"reports/daily_coverage.csv",index=False)
        daily[["signal_date","model","score_min","score_max","score_std","informative"]].to_csv(run/"reports/score_quality.csv",index=False)
        pd.concat(exclusions,ignore_index=True).to_csv(run/"reports/exclusions.csv",index=False)
        daily.loc[~daily.calendar_available].to_csv(run/"reports/calendar_gaps.csv",index=False)
        save_json_atomic(run/"reports/source_validation.json",dict(status="pass",prediction_hashes_verified=source["verified_hashes"],
            forbidden_fields_in_decisions=False,calendar_end=str(market["calendar"].max().date())))
        metadata.update(status="complete",rows_by_model=counts,total_rows=sum(counts.values()),
                        partitions=len(inventory),files_sha256=inventory,
                        completed_at=datetime.now(timezone.utc).isoformat())
        save_json_atomic(run/"metadata.json",metadata)
        save_json_atomic(root/"latest_run.json",{"run_dir":run.name})
    except BaseException as exc:
        metadata.update(status="failed",error=f"{type(exc).__name__}: {exc}")
        save_json_atomic(run/"metadata.json",metadata)
        raise
    return run


def iter_signal_batches(run_dir, model=None, start=None, end=None, verify_hashes=False):
    run = Path(run_dir)
    meta = read_json_file(run/"metadata.json")
    if meta.get("schema") != SIGNAL_SCHEMA or meta.get("status") != "complete":
        raise ValueError("请选择成功完成的信号运行。")
    if stable_hash(read_json_file(run/"config.json")) != meta["config_hash"]:
        raise ValueError("信号配置校验失败。")
    names = [model] if model else meta["models"]
    for name in names:
        if name not in meta["models"]:
            raise ValueError(f"未知模型：{name}")
        for path in sorted((run/"signals"/f"model={name}").glob("part-*.parquet")):
            month = pd.Timestamp(path.stem.removeprefix("part-")+"01")
            if start is not None and month+pd.offsets.MonthEnd(0) < pd.Timestamp(start):
                continue
            if end is not None and month > pd.Timestamp(end):
                continue
            if verify_hashes:
                require_hash(path,meta["files_sha256"][str(path.relative_to(run))])
            frame = pd.read_parquet(path)
            if start is not None: frame = frame.loc[frame.signal_date.ge(pd.Timestamp(start))]
            if end is not None: frame = frame.loc[frame.signal_date.le(pd.Timestamp(end))]
            if len(frame): yield frame


def load_signal_data(run_dir, model=None, start=None, end=None, verify_hashes=False):
    pieces = list(iter_signal_batches(run_dir,model,start,end,verify_hashes))
    if not pieces:
        return pd.DataFrame()
    frame = pd.concat(pieces,ignore_index=True)
    if frame.duplicated(["signal_date","instrument","model"]).any():
        raise ValueError("已导出信号出现重复主键。")
    return frame.sort_values(["signal_date","model","score_rank","instrument"],na_position="last").reset_index(drop=True)


In [10]:
def reusable_signal_run(root, source, market, config):
    if not (Path(root) / "latest_run.json").exists():
        return None
    run = resolve_latest(root)
    meta = read_json_file(run / "metadata.json")
    if meta.get("status") != "complete" or meta.get("schema") != SIGNAL_SCHEMA:
        return None
    if meta.get("source_hash") != source["source_hash"] or meta.get("config_hash") != stable_hash(config):
        return None
    saved = read_json_file(run / "source.json")
    if stable_hash(saved["market_files"]) != stable_hash(market["source_files"]):
        return None
    for relative, digest in meta["files_sha256"].items():
        require_hash(run / relative, digest)
    return run

SIGNAL_RUN = reusable_signal_run(OUTPUT_ROOT, source, market, SIGNAL_CONFIG) if REUSE_MATCHING_RUN else None
if SIGNAL_RUN is None:
    if not RUN_EXPORT:
        raise RuntimeError("没有同输入/配置的成功版本；先将 RUN_EXPORT=True 运行导出。")
    SIGNAL_RUN = export_signal_dataset(source, market, SIGNAL_CONFIG, OUTPUT_ROOT)
else:
    print("来源、行情、配置和分区校验通过，复用：", SIGNAL_RUN)

signal_metadata = read_json_file(SIGNAL_RUN / "metadata.json")
print("信号运行：", SIGNAL_RUN)
print("总行数：", f"{signal_metadata['total_rows']:,}", "；结果等级：", signal_metadata["result_level"])
coverage_report = pd.read_csv(SIGNAL_RUN / "reports/daily_coverage.csv")
display(coverage_report.groupby("model").agg(days=("signal_date", "size"), rows=("rows", "sum"),
    actionable=("actionable", "sum"), informative_days=("informative", "sum")))
display(pd.read_csv(SIGNAL_RUN / "reports/calendar_gaps.csv").head(6))


来源、行情、配置和分区校验通过，复用： E:\document-prefer\python for finanace\multi-model-bactest\data\processed\signals\run_20261025T142048Z_4769f72d
信号运行： E:\document-prefer\python for finanace\multi-model-bactest\data\processed\signals\run_20261025T142048Z_4769f72d
总行数： 7,455,606 ；结果等级： research_only


,days,rows,actionable,informative_days
model,,,,
lasso,243,1242601,1237459,242
lightgbm,243,1242601,1237459,242
mlp,243,1242601,1237459,242
ols,243,1242601,1237459,242
random_forest,243,1242601,1237459,242
xgboost,243,1242601,1237459,242


,signal_date,model,rows,candidates,actionable,informative,score_min,score_max,score_std,calendar_available
0,2025-12-31,ols,5142,0,0,False,-0.010260,0.024909,0.002526,False
1,2025-12-31,lasso,5142,0,0,False,0.001210,0.003542,0.000244,False
2,2025-12-31,random_forest,5142,0,0,False,-0.015141,0.036890,0.001499,False
3,2025-12-31,xgboost,5142,0,0,False,-0.017250,0.027500,0.001441,False
4,2025-12-31,lightgbm,5142,0,0,False,-0.008451,0.026028,0.001270,False
5,2025-12-31,mlp,5142,0,0,False,-0.015667,0.037370,0.003707,False


## 7. 定义账户快照：组合必须知道实际持仓

**输入**：`account_id/model/strategy_id/as_of`，可用与预留现金、NAV，以及每只股票的总股数、可卖股数、冻结股数、估值和估值时间。

**为什么需要**：昨天的目标不一定成交。若卖单失败，股票和资金仍在旧状态；用昨天目标代替今天真实持仓，会虚构现金和新席位。

下面的校验要求现金加持仓估值能够与 NAV 核对；同一模型和策略只能使用自己的账户；快照和价格不得晚于决策时点。组合函数读取快照后不会修改它。

`empty_account_snapshot` 仅构造**示例空仓账户**。`one_day_market` 返回 t 日已知的价格快照，不读取 t+1 收盘价。接入真实引擎时用实际成交后账户替代示例快照。

In [11]:
def as_shanghai(value):
    value = pd.Timestamp(value)
    if pd.isna(value):
        raise ValueError("决策、账户或价格的可用时点不能缺失。")
    return value.tz_localize("Asia/Shanghai") if value.tzinfo is None else value.tz_convert("Asia/Shanghai")


def strategy_identity(model, config):
    return config.get("strategy_id", f"{config.get('policy','topk_buffer')}_v1__{model}")


def validate_account(snapshot, model, config, decision_at):
    if snapshot.get("model") != model or snapshot.get("strategy_id") != strategy_identity(model,config):
        raise ValueError("账户模型/策略标识不一致，禁止交叉使用持仓。")
    if as_shanghai(snapshot["as_of"]) > decision_at:
        raise ValueError("账户快照晚于决策时间。")
    positions = snapshot["positions"].copy()
    if "instrument" in positions.columns:
        positions = positions.set_index("instrument")
    needed = ["quantity","sellable_quantity","frozen_quantity","valuation_price","price_as_of"]
    if positions.index.has_duplicates or not set(needed).issubset(positions.columns):
        raise ValueError("持仓字段缺失或代码重复。")
    for name in ["quantity","sellable_quantity","frozen_quantity"]:
        values = pd.to_numeric(positions[name],errors="coerce")
        if not np.isfinite(values).all() or (values<0).any() or not np.equal(values,np.floor(values)).all():
            raise ValueError("持仓数量必须为非负整数。")
        positions[name] = values.astype("int64")
    if (positions.sellable_quantity+positions.frozen_quantity > positions.quantity).any():
        raise ValueError("可卖与冻结数量超过总持仓。")
    positions = positions.loc[positions.quantity.gt(0)].copy()
    if len(positions):
        if not np.isfinite(positions.valuation_price).all() or (positions.valuation_price<=0).any():
            raise ValueError("已有持仓缺少可核对估值。")
        if any(as_shanghai(t)>decision_at for t in positions.price_as_of):
            raise ValueError("持仓估值使用了未来价格。")
    cash = float(snapshot["cash_available"])
    reserved, nav = float(snapshot.get("cash_reserved",0)), float(snapshot["nav"])
    if not np.isfinite([cash,reserved,nav]).all() or min(cash,reserved)<0 or nav<=0:
        raise ValueError("账户现金/NAV 无效。")
    calculated = cash+reserved+float((positions.quantity*positions.valuation_price).sum())+float(snapshot.get("other_assets",0))
    if not np.isclose(calculated,nav,rtol=1e-7,atol=0.02):
        raise ValueError("NAV 与现金、预留资金、持仓估值不一致。")
    return positions


def empty_account_snapshot(model, decision_at, config, initial_cash=1_000_000):
    columns = ["quantity","sellable_quantity","frozen_quantity","valuation_price","price_as_of"]
    return dict(account_id=f"preview_{model}",strategy_id=strategy_identity(model,config),model=model,
                as_of=as_shanghai(decision_at),cash_available=float(initial_cash),cash_reserved=0.0,
                nav=float(initial_cash),positions=pd.DataFrame(columns=columns).rename_axis("instrument"),
                open_orders=pd.DataFrame(),snapshot_role="illustrative_empty_account")


def account_snapshot_hash(snapshot, positions):
    return stable_hash(dict(account_id=snapshot["account_id"],as_of=str(snapshot["as_of"]),
        nav=float(snapshot["nav"]),cash=float(snapshot["cash_available"]),
        reserved=float(snapshot.get("cash_reserved",0)),positions=positions.reset_index().to_dict("records"),
        open_orders=snapshot.get("open_orders",pd.DataFrame()).to_dict("records"),
        replacement_counter_date=str(snapshot.get("replacement_counter_date")),
        discretionary_replacements_used=int(snapshot.get("discretionary_replacements_used",0))))


def one_day_market(market, day):
    table = market["daily"].xs(pd.Timestamp(day),level="signal_date").copy()
    table["price_as_of"] = as_shanghai(pd.Timestamp(day)+pd.Timedelta(hours=15))
    table["as_of"] = table["price_as_of"]
    return table


## 8. 生成目标权重：等权与换手缓冲

**输入**：一个交易日、一个模型的排名，真实账户快照和当时市场快照。

两种策略共用相同资格条件：`topk_equal` 对目标前 K 规划等权；`topk_buffer` 尽量保留旧仓股数，只提出有限的完整退出与新席位。

缓冲策略依次处理：

1. 空仓首次可规划最多 K 个新席位；已有持仓和未成交买单都占用席位。
2. 缺分数、不可卖、冻结或有未完成订单的旧仓保留，并占用预算。
3. 对已跌出前 K、分数可用且可完整卖出的旧仓，从最差者开始提出退出，正常最多 5 个。
4. 依赖旧仓退出的新候选标为 `defer/requires_exit_and_cash_confirmation`，其立即可执行权重为零。尚未成交的卖单不提供现金或席位。
5. 已有仓位超预算时明确记录违反状态，暂停扩大风险的新买入。候选不足时保留现金，不自动放大单只权重。

**输出**：`targets` 和 `decision_log`。`desired_weight` 是理想意图，`constrained_target_weight` 加入已知限制。保留持仓也有明确一行，不靠缺行表示清仓。

同一信号日若成交后再次调用，账户应提供 `replacement_counter_date` 和 `discretionary_replacements_used`，避免重置当日轮换预算；本函数不替账户维护计数。

In [12]:
def build_target_weights(ranked_signals, account_snapshot, known_market_snapshot, config):
    """纯策略函数：接收真实账户，不修改账户，不假定交易已成交。"""
    signals = ranked_signals.copy()
    if "instrument" not in signals.columns:
        signals = signals.reset_index()
    if FORBIDDEN_DECISION_FIELDS.intersection(signals.columns):
        raise ValueError("组合接口禁止接收未来标签。")
    if signals.empty or signals.model.nunique()!=1 or signals.signal_date.nunique()!=1:
        raise ValueError("一次调用必须是非空的单日、单模型信号。")
    if signals.instrument.duplicated().any():
        raise ValueError("单日信号股票重复。")
    model = signals.model.iloc[0]
    decision_at = as_shanghai(config.get("decision_at",signals.signal_available_at.max()))
    if signals.signal_available_at.map(as_shanghai).gt(decision_at).any():
        raise ValueError("信号在决策时尚未可用。")
    positions = validate_account(account_snapshot,model,config,decision_at)
    known = known_market_snapshot.copy()
    if "instrument" in known.columns: known=known.set_index("instrument")
    if known.index.has_duplicates:
        raise ValueError("市场快照代码重复。")
    if "as_of" not in known or known["as_of"].map(as_shanghai).gt(decision_at).any():
        raise ValueError("市场快照时间未知或包含未来信息。")
    k, drop = int(config.get("top_k",50)), int(config.get("max_discretionary_replacements",5))
    budget, cap = float(config.get("target_stock_weight",0.95)), float(config.get("single_name_weight_cap",0.03))
    policy = config.get("policy","topk_buffer")
    if policy not in {"topk_equal","topk_buffer"} or k<1 or drop<0 or not 0<budget<=1 or not 0<cap<=1:
        raise ValueError("组合配置无效。")
    mode = config.get("status_mode","research")
    if mode not in {"research","strict"}:
        raise ValueError("未知的状态模式。")
    by_code = signals.set_index("instrument")
    informative = bool(signals.cross_section_informative.all())
    execution_day = signals.earliest_execution_date.iloc[0]
    can_plan = informative and pd.notna(execution_day)
    candidates = signals.loc[signals.candidate_eligible.eq(True)].sort_values(
        ["score_raw","instrument"],ascending=[False,True]) if can_plan else signals.iloc[:0]
    if mode=="strict" and len(candidates):
        eligibility=known.reindex(candidates.instrument)
        eligible=(eligibility.get("historical_status_verified",pd.Series(False,index=eligibility.index)).fillna(False).astype(bool)
                  & eligibility.get("known_can_buy",pd.Series(False,index=eligibility.index)).fillna(False).astype(bool)
                  & eligibility.get("market_observed",pd.Series(False,index=eligibility.index)).fillna(False).astype(bool))
        candidates=candidates.loc[eligible.to_numpy()]
        if len(candidates)<config.get("min_cross_section",30) or candidates.score_raw.nunique()<2:
            candidates=signals.iloc[:0]
            can_plan=False
    top = candidates.head(k)
    top_codes, held = set(top.instrument), set(positions.index)
    pending = set()
    open_orders = account_snapshot.get("open_orders",pd.DataFrame())
    if len(open_orders):
        if not {"instrument","status"}.issubset(open_orders.columns):
            raise ValueError("未完成订单缺少 instrument/status。")
        active = open_orders.status.isin(["pending","open","partially_filled","submitted"])
        pending = set(open_orders.loc[active,"instrument"])
    def can_fully_exit(code):
        pos = positions.loc[code]
        if pos.sellable_quantity < pos.quantity or pos.frozen_quantity>0 or code in pending or code not in known.index:
            return False
        row = known.loc[code]
        if not bool(row.get("market_observed",False)):
            return False
        if bool(row.get("historical_status_verified",False)):
            return bool(row.get("known_can_sell",False))
        return mode=="research"
    exit_candidates=[]
    if can_plan:
        for code in held-top_codes:
            if code in by_code.index and np.isfinite(by_code.loc[code,"score_raw"]) and can_fully_exit(code):
                exit_candidates.append(code)
    exit_candidates.sort(key=lambda code:(float(by_code.loc[code,"score_raw"]),str(code)))
    counter_day=account_snapshot.get("replacement_counter_date")
    used=0
    if counter_day is not None:
        if pd.Timestamp(counter_day)>pd.Timestamp(signals.signal_date.iloc[0]):
            raise ValueError("账户轮换计数来自未来日期。")
        if pd.Timestamp(counter_day)==pd.Timestamp(signals.signal_date.iloc[0]):
            used=int(account_snapshot.get("discretionary_replacements_used",0))
        if used<0:raise ValueError("已使用的主动轮换计数不能为负。")
    remaining_drop=max(0,drop-used)
    exits = set(exit_candidates if policy=="topk_equal" else exit_candidates[:remaining_drop])
    current_value = float((positions.quantity*positions.valuation_price).sum())
    nav=float(account_snapshot["nav"])
    current_stock_weight=current_value/nav
    existing_breach = current_stock_weight>budget+1e-10 or bool(((positions.quantity*positions.valuation_price)/nav>cap+1e-10).any())
    # 未成交的新买单也占用席位，避免重复调用时超出 K。
    free_slots=max(0,k-len(held)-len(pending-held))
    # 排名落后但尚未真实退出的股票不提供新的现金或席位。
    newcomers = candidates.loc[~candidates.instrument.isin(held|pending)].head(free_slots+len(exits))
    slot=min(budget/k,cap)
    headroom=max(0.0,budget-current_stock_weight)
    rows, logs=[],[]
    common=dict(signal_date=signals.signal_date.iloc[0],decision_at=decision_at,
        earliest_execution_date=execution_day,account_id=account_snapshot["account_id"],
        strategy_id=account_snapshot["strategy_id"],model=model,result_level=signals.result_level.iloc[0],
        source_hash=signals.source_hash.iloc[0],config_hash=stable_hash(config),
        snapshot_hash=account_snapshot_hash(account_snapshot,positions))
    for code,pos in positions.sort_index().iterrows():
        weight=float(pos.quantity*pos.valuation_price/nav)
        intent, reason, target, preserve="hold","retained_quantity",weight,True
        if code in pending:
            reason="open_order_pending"
        elif code in exits:
            intent,reason,target,preserve="exit","exit_rank_outside_topk",0.0,False
        elif code not in by_code.index or not np.isfinite(by_code.loc[code,"score_raw"]):
            reason="held_missing_signal"
        elif not can_plan:
            reason="cross_section_or_calendar_disabled"
        elif not can_fully_exit(code):
            reason="held_not_fully_sellable"
        elif policy=="topk_equal" and code in top_codes:
            locked_weight=max(0,int(pos.quantity)-int(pos.sellable_quantity))*float(pos.valuation_price)/nav
            target=max(slot,locked_weight)
            intent="rebalance"
            reason="equal_weight_target"
            preserve=False
        rows.append(dict(**common,instrument=code,current_quantity=int(pos.quantity),sellable_quantity=int(pos.sellable_quantity),
            current_weight=weight,desired_weight=0.0 if code in exits else (slot if code in top_codes else weight),
            constrained_target_weight=target,target_quantity_estimate=int(pos.quantity) if preserve else np.nan,
            preserve_quantity=preserve,intent=intent,condition="none",feasible=pd.notna(execution_day),decision_reason=reason))
    for number, row in enumerate(newcomers.itertuples(index=False)):
        unconditional = number<free_slots and headroom>0 and not existing_breach
        weight=min(slot,headroom) if unconditional else 0.0
        if unconditional: headroom-=weight
        reason="open_empty_slot" if unconditional else ("existing_constraint_breach" if existing_breach else "requires_exit_fill")
        rows.append(dict(**common,instrument=row.instrument,current_quantity=0,sellable_quantity=0,current_weight=0.0,
            desired_weight=slot,constrained_target_weight=weight,target_quantity_estimate=np.nan,
            preserve_quantity=False,intent="open" if unconditional else "defer",
            condition="none" if unconditional else "requires_exit_and_cash_confirmation",feasible=unconditional,decision_reason=reason))
    if not rows:
        logs.append(dict(model=model,decision_at=decision_at,reason="no_actionable_candidate_or_position"))
    if existing_breach:
        logs.append(dict(model=model,decision_at=decision_at,reason="constraint_breach_existing_position"))
    targets=pd.DataFrame(rows)
    if len(targets):
        logs += targets[["instrument","decision_reason","feasible"]].rename(columns={"decision_reason":"reason"}).to_dict("records")
    return targets,pd.DataFrame(logs)


## 9. 估算目标股数与资金预算

**输入**：上一步目标、同一账户、当时已知的未复权参考价格和规则表。

**处理**：目标金额按已知 NAV 计算，再根据参考价格计算增减数量。买单按可用资金扣除现金缓冲和预估费用限额；同批卖出收入不加入预算。处理顺序固定为意图与股票代码，保证复现。

股数合法化针对**订单增减数量**，不是简单把最终持仓取整：已有零股时，追加买入仍应满足买单规则。研究模式提供标记清楚的主板/创业板与科创板数量假设；strict 模式缺少已验证的有效规则则不给新买单。正式历史规则需要由回测数据适配层提供。

**输出**：`quantity_intents` 和校验日志。`quantity_delta>0` 是拟买入、`<0` 是拟卖出、`=0` 是不产生订单。这里仅预算买入费用；实际税费和账户变动在成交层记一次。

目标表同时绑定账户快照和配置的哈希。若现金、持仓、未完成订单或策略配置发生变化，必须基于新状态重新生成目标，不能拿旧目标套用到新账户。

未来价格被禁止。实际次日价格不同、涨跌停或订单未成交时，后续执行层要再校验，不能把这里的意图当作已成交持仓。

In [13]:
def rule_for_instrument(code, day, rules, mode):
    board="star" if code.endswith(".SH") and code.startswith(("688","689")) else (
        "gem" if code.endswith(".SZ") and code.startswith(("300","301")) else "main")
    if rules is not None and len(rules):
        table=rules.copy()
        mask=table["board"].eq(board)&pd.to_datetime(table["effective_from"]).le(day)
        mask &= pd.to_datetime(table["effective_to"]).fillna(pd.Timestamp.max).ge(day)
        selected=table.loc[mask]
        if len(selected)!=1:
            raise ValueError("生效规则缺失或重叠。")
        rule=selected.iloc[0].to_dict()
        if mode=="strict" and not bool(rule.get("verified",False)):
            return None
        return rule
    if mode=="strict": return None
    return dict(buy_min=200 if board=="star" else 100,buy_step=1 if board=="star" else 100,
                sell_min=200 if board=="star" else 100,sell_step=1 if board=="star" else 100,
                verified=False,board=board)


def legal_order_quantity(raw, minimum, step):
    if raw<minimum: return 0
    return int(minimum+np.floor((raw-minimum)/step)*step)


def estimate_target_shares(targets,account_snapshot,known_price_snapshot,rules=None,config=None):
    """预估订单数量和预算；不计入卖出收入、不扣实际现金、不写成交。"""
    config=dict(config or {})
    if targets.empty:return targets.copy(),pd.DataFrame()
    if targets.model.nunique()!=1 or targets.decision_at.nunique()!=1:
        raise ValueError("股数接口一次只能处理同一模型和决策时点。")
    if not targets.account_id.eq(account_snapshot["account_id"]).all() or not targets.strategy_id.eq(account_snapshot["strategy_id"]).all():
        raise ValueError("目标组合与账户标识不一致。")
    model=targets.model.iloc[0]
    decision_at=as_shanghai(targets.decision_at.iloc[0])
    positions=validate_account(account_snapshot,model,config,decision_at)
    if not targets.config_hash.eq(stable_hash(config)).all():
        raise ValueError("股数估算配置与目标配置不一致，请重新生成目标。")
    if not targets.snapshot_hash.eq(account_snapshot_hash(account_snapshot,positions)).all():
        raise ValueError("账户快照已变化，请根据新快照重新生成目标。")
    actual_quantities=positions.quantity.reindex(targets.instrument).fillna(0).to_numpy()
    if not np.array_equal(actual_quantities,targets.current_quantity.to_numpy()):
        raise ValueError("目标引用的持仓已经过期，请使用最新账户重新生成目标。")
    prices=known_price_snapshot.copy()
    if "instrument" in prices.columns:prices=prices.set_index("instrument")
    if prices.index.has_duplicates or not {"close","price_as_of"}.issubset(prices.columns):
        raise ValueError("价格快照字段缺失或代码重复。")
    if prices.price_as_of.map(as_shanghai).gt(decision_at).any():
        raise ValueError("股数估算使用了未来价格。")
    mode=config.get("status_mode","research")
    reserve=max(0.0,1-float(config.get("target_stock_weight",0.95)))*float(account_snapshot["nav"])
    remaining=max(0.0,float(account_snapshot["cash_available"])-reserve)
    commission=float(config.get("commission_rate_estimate",0.0003))
    minimum_fee=float(config.get("min_commission_estimate",5.0))
    other_rate=float(config.get("other_buy_fee_rate_estimate",0.0))
    if min(commission,minimum_fee,other_rate)<0:
        raise ValueError("预估费率不能为负。")
    def cost(quantity,price):
        notional=quantity*price
        return notional+(max(notional*commission,minimum_fee) if quantity else 0)+notional*other_rate
    output,logs=[],[]
    # 只排序意图处理顺序，绝不把同批卖出收入加入 remaining。
    for row in targets.sort_values(["intent","instrument"]).to_dict("records"):
        code=row["instrument"]; current=int(row["current_quantity"])
        delta,fee,notional,price=0,0.0,0.0,np.nan
        reason=row["decision_reason"]
        if row["preserve_quantity"] or not row["feasible"] or row["intent"] in {"hold","defer"}:
            pass
        elif code not in prices.index or not np.isfinite(prices.loc[code,"close"]) or prices.loc[code,"close"]<=0:
            reason="missing_known_reference_price"
        else:
            price=float(prices.loc[code,"close"])
            rule=rule_for_instrument(code,pd.Timestamp(row["signal_date"]),rules,mode)
            if rule is None:
                reason="historical_quantity_rule_unknown"
            else:
                desired=float(row["constrained_target_weight"])*float(account_snapshot["nav"])/price
                raw_delta=desired-current
                if raw_delta>0:
                    quantity=legal_order_quantity(raw_delta,int(rule["buy_min"]),int(rule["buy_step"]))
                    affordable=max(0.0,(remaining-minimum_fee)/(price*(1+commission+other_rate)))
                    quantity=min(quantity,legal_order_quantity(affordable,int(rule["buy_min"]),int(rule["buy_step"])))
                    while quantity>0 and cost(quantity,price)>remaining+1e-9:
                        quantity=legal_order_quantity(quantity-int(rule["buy_step"]),int(rule["buy_min"]),int(rule["buy_step"]))
                    delta=quantity
                    notional=delta*price
                    fee=max(notional*commission,minimum_fee)+notional*other_rate if delta else 0.0
                    remaining-=notional+fee
                    if delta==0:reason="cash_or_minimum_quantity_limit"
                elif raw_delta<0:
                    sellable=int(positions.loc[code,"sellable_quantity"]) if code in positions.index else 0
                    requested=min(float(-raw_delta),sellable)
                    quantity=current if desired<=1e-8 and sellable>=current else legal_order_quantity(requested,int(rule["sell_min"]),int(rule["sell_step"]))
                    delta=-quantity;notional=quantity*price
                    # 卖出税费留给后续统一成交费用模块，不作为本批买入资金。
                row["quantity_rule_verified"]=bool(rule.get("verified",False))
        row.update(target_quantity_estimate=current+delta,quantity_delta=delta,reference_price=price,
                   reference_price_as_of=prices.loc[code,"price_as_of"] if code in prices.index else pd.NaT,
                   estimated_buy_fee=fee,estimated_buy_cash=notional+fee if delta>0 else 0.0,
                   intent_status="quantity_intent_only" if delta else "no_order",
                   decision_reason=reason,cash_budget_remaining=remaining)
        output.append(row);logs.append(dict(instrument=code,quantity_delta=delta,reason=reason,remaining_cash_budget=remaining))
    if remaining < -0.01:
        raise AssertionError("预估买单超过可用资金预算。")
    return pd.DataFrame(output),pd.DataFrame(logs)


## 10. 用真实首日信号演示六模型目标组合

**输入**：导出后重新读取的首个信号日分数，以及每个模型单独构造的空仓快照。

**处理**：对同一天、同样的 100 万示例资金分别执行六模型。每个模型展示其目标数、非零股数意图及预算；不比较这一天的未来收益。分数恒定的模型可能没有新买入，这是资格与信息量规则的正常结果。

**输出**：`previews/preview_.../` 中的目标、股数意图、原因和示例元数据，角色明确为 `idealized_preview`。它们是一次日期快照，不是用“全部成交”假设递推出来的一年持仓。

你可以改变 `PREVIEW_DAY` 和 `PREVIEW_INITIAL_CASH` 观察单位、预算及最低申报的影响。生成新预览目录，不覆盖原始信号。

In [14]:
preview_stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
PREVIEW_DIR = SIGNAL_RUN / "previews" / f"preview_{preview_stamp}_{uuid.uuid4().hex[:8]}"
PREVIEW_DIR.mkdir(parents=True, exist_ok=False)
known_day = one_day_market(market, PREVIEW_DAY)
first_day_frames = {}
preview_summary = []

for model in source["models"]:
    day_signals = load_signal_data(SIGNAL_RUN, model=model, start=PREVIEW_DAY, end=PREVIEW_DAY,
                                   verify_hashes=True)
    first_day_frames[model] = day_signals
    snapshot = empty_account_snapshot(model, day_signals.signal_available_at.max(),
                                       PORTFOLIO_CONFIG, PREVIEW_INITIAL_CASH)
    targets, reasons = build_target_weights(day_signals, snapshot, known_day, PORTFOLIO_CONFIG)
    quantities, quantity_checks = estimate_target_shares(targets, snapshot, known_day,
                                                        rules=None, config=PORTFOLIO_CONFIG)
    if len(targets):
        targets.to_parquet(PREVIEW_DIR / f"{model}_targets.parquet", index=False)
        quantities.to_parquet(PREVIEW_DIR / f"{model}_quantity_intents.parquet", index=False)
    reasons.to_csv(PREVIEW_DIR / f"{model}_decisions.csv", index=False)
    preview_summary.append({
        "model": model, "targets": len(targets),
        "nonzero_quantity_intents": int(quantities.quantity_delta.ne(0).sum()) if len(quantities) else 0,
        "estimated_buy_cash": float(quantities.estimated_buy_cash.sum()) if len(quantities) else 0.0,
        "role": "idealized_preview",
    })

preview_summary = pd.DataFrame(preview_summary)
preview_summary.to_csv(PREVIEW_DIR / "summary.csv", index=False)
save_json_atomic(PREVIEW_DIR / "metadata.json", {
    "role": "idealized_preview", "signal_run": str(SIGNAL_RUN), "signal_date": PREVIEW_DAY,
    "initial_cash_example": PREVIEW_INITIAL_CASH, "config": PORTFOLIO_CONFIG,
    "account_source": "explicit_empty_snapshot", "assumed_fills": False,
})
display(preview_summary)
example_path = PREVIEW_DIR / "ols_quantity_intents.parquet"
if example_path.exists():
    display(pd.read_parquet(example_path)[["instrument", "desired_weight", "constrained_target_weight",
        "quantity_delta", "reference_price", "estimated_buy_cash", "decision_reason"]].head(10))
print("预览目录：", PREVIEW_DIR)


,model,targets,nonzero_quantity_intents,estimated_buy_cash,role
0,ols,50,50,905027.953883,idealized_preview
1,lasso,50,48,875376.049676,idealized_preview
2,random_forest,50,50,914086.693240,idealized_preview
3,xgboost,50,50,914407.789540,idealized_preview
4,lightgbm,50,50,912253.143340,idealized_preview
5,mlp,50,50,893045.671948,idealized_preview


,instrument,desired_weight,constrained_target_weight,quantity_delta,reference_price,estimated_buy_cash,decision_reason
0,000788.SZ,0.019,0.019,3200,5.83,18661.5968,open_empty_slot
1,000818.SZ,0.019,0.019,1000,17.47,17475.2410,open_empty_slot
2,000833.SZ,0.019,0.019,1600,11.37,18197.4576,open_empty_slot
3,002122.SZ,0.019,0.019,5500,3.44,18925.6760,open_empty_slot
4,002137.SZ,0.019,0.019,2100,8.73,18338.4999,open_empty_slot
5,002329.SZ,0.019,0.019,5300,3.52,18661.5968,open_empty_slot
6,002347.SZ,0.019,0.019,2800,6.75,18905.6700,open_empty_slot
7,002354.SZ,0.019,0.019,3500,5.39,18870.6595,open_empty_slot
8,002469.SZ,0.019,0.019,2600,7.19,18699.6082,open_empty_slot
9,002512.SZ,0.019,0.019,4200,4.48,18821.6448,open_empty_slot


预览目录： E:\document-prefer\python for finanace\multi-model-bactest\data\processed\signals\run_20261025T142048Z_4769f72d\previews\preview_20261025T142711Z_c71b8053


## 11. 可选的等权排名融合

六模型独立策略是默认基线。这里额外提供一个纯函数，将同日共同合格股票池内的六模型分别转成排名分位，再等权平均。

**为什么重算共同池排名**：不同模型的原始分数尺度可能不同，缺失覆盖也可能不同。直接相加原始分数会让某个模型因数值尺度较大而占更高权重。

**边界**：必须六模型完整、日期与来源相同；任一成员在共同池中恒定或覆盖不足，融合不产生新增买入；缺模型不会静默补零或临时改权重。融合规则与未来收益无关。

下面只定义接口。`RUN_ENSEMBLE_PREVIEW` 默认关闭，开启只显示这个日期的融合示例；不将它自动替换成六模型基线。

In [15]:
def build_rank_ensemble_day(model_frames, config):
    """可选、单日等权排名融合；共同资格池、固定六模型，无标签或收益选权。"""
    if set(model_frames)!=set(EXPECTED_MODELS):
        raise ValueError("排名融合必须提供固定的六个模型，不能静默少一个。")
    frames={name:frame.copy() for name,frame in model_frames.items()}
    for name,frame in frames.items():
        if frame.empty or frame.model.nunique()!=1 or frame.model.iloc[0]!=name or frame.signal_date.nunique()!=1:
            raise ValueError("融合输入必须为各模型的同一交易日。")
        if frame.instrument.duplicated().any() or FORBIDDEN_DECISION_FIELDS.intersection(frame.columns):
            raise ValueError("融合输入重复或包含标签。")
    dates={frame.signal_date.iloc[0] for frame in frames.values()}
    source_hashes={frame.source_hash.iloc[0] for frame in frames.values()}
    if len(dates)!=1 or len(source_hashes)!=1:
        raise ValueError("融合日期或源运行不一致。")
    common=None
    for frame in frames.values():
        eligible=set(frame.loc[frame.candidate_eligible.eq(True)&np.isfinite(frame.score_raw),"instrument"])
        common=eligible if common is None else common&eligible
    if not common:
        raise ValueError("六模型共同合格股票池为空。")
    codes=sorted(common)
    first=frames[EXPECTED_MODELS[0]].set_index("instrument").loc[codes].copy()
    first["model"]="ensemble_rank"
    member_values=[]
    member_informative=True
    for name in EXPECTED_MODELS:
        frame=frames[name].set_index("instrument").loc[codes]
        n=len(frame)
        rank=frame.score_raw.rank(method="average",ascending=True)
        member_values.append(((rank-1)/(n-1)).rename(name))
        member_informative &= bool(n>=config.get("min_cross_section",30) and frame.score_raw.nunique()>1)
        if not frame.earliest_execution_date.equals(first.earliest_execution_date):
            raise ValueError("各模型的最早执行日期不一致。")
    members=pd.concat(member_values,axis=1)
    first["score_raw"]=members.mean(axis=1)
    first["candidate_eligible"]=bool(member_informative)
    first["reason_code"]="ensemble_common_pool" if member_informative else "ensemble_uninformative_member"
    first["signal_available_at"]=max(f.signal_available_at.max() for f in frames.values())
    first["model_available_after"]=max(f.model_available_after.max() for f in frames.values())
    first["max_train_label_end"]=max(f.max_train_label_end.max() for f in frames.values())
    first=first.drop(columns=[c for c in ["score_rank","score_percentile","candidate_count","distinct_scores",
        "cross_section_informative","actionable","cross_section_status"] if c in first.columns])
    first["ensemble_members"]="|".join(EXPECTED_MODELS)
    first["ensemble_rule"]="equal_rank_on_common_pool"
    return rank_signals(first.reset_index(),config),members


In [16]:
RUN_ENSEMBLE_PREVIEW = False
if RUN_ENSEMBLE_PREVIEW:
    ensemble_day, member_ranks = build_rank_ensemble_day(first_day_frames, PORTFOLIO_CONFIG)
    display(ensemble_day[["instrument", "score_raw", "score_rank", "actionable", "reason_code"]].head(10))
else:
    print("融合接口已定义；默认保持六模型独立基线。")


融合接口已定义；默认保持六模型独立基线。


## 12. 自动验收：检查规则，而不是挑选收益

这一格用小型合成数据和可人工核对的账户快照检查边界：并列排序、禁止标签、常数分数、日历末端、空仓建仓、冻结旧仓、卖单未成交、同日轮换计数、待成交买单、未来价格、账户串用、板块数量和融合缺模型等。

还会在明确的临时目录里做一轮小文件保存/加载，并故意制造一个失败导出，确认失败不会覆盖之前的成功 latest。临时路径经过范围校验后清理。

**输出**：每项检查名称与 pass。这里的测试通过表示接口和资金/时间边界符合约定，不表示任何模型未来盈利。失败时先修复对应边界，再继续接交易引擎。

In [17]:
def run_signal_acceptance_checks(work_dir=None):
    """用小型手工场景验证时间和资金边界，不用策略收益判定通过。"""
    import copy
    import tempfile
    records=[]
    cfg=dict(status_mode="research",min_cross_section=30,policy="topk_buffer",top_k=50,
             max_discretionary_replacements=5,target_stock_weight=0.95,single_name_weight_cap=0.03,
             commission_rate_estimate=0.0003,min_commission_estimate=5.0)
    day=pd.Timestamp("2025-01-02")
    at=as_shanghai("2025-01-02 15:30")
    codes=[f"{i+1:06d}.SZ" for i in range(70)]
    raw=pd.DataFrame(dict(signal_date=day,instrument=codes,model="ols",score_raw=np.arange(70,0,-1,dtype=float),
        candidate_eligible=True,earliest_execution_date=pd.Timestamp("2025-01-03"),
        signal_available_at=at,result_level="research_only",source_hash="fixture",
        model_available_after=pd.Timestamp("2024-12-31"),max_train_label_end=pd.Timestamp("2024-12-31")))
    ranked=rank_signals(raw,cfg)
    known=pd.DataFrame(dict(instrument=codes,close=10.0,market_observed=True,historical_status_verified=False,
                           known_can_sell=False,known_can_buy=False,as_of=at,price_as_of=at)).set_index("instrument")
    def record(name, condition):
        if not condition:raise AssertionError(name)
        records.append(dict(check=name,status="pass"))
    def rejects(name, function):
        try:function()
        except (ValueError,KeyError):records.append(dict(check=name,status="pass"));return
        raise AssertionError(name)
    # 同分普通并列保持确定性；输入次序改变不改变排序。
    tied=raw.copy();tied.loc[:2,"score_raw"]=100.0
    a=rank_signals(tied,cfg).set_index("instrument")["score_rank"]
    b=rank_signals(tied.sample(frac=1,random_state=7),cfg).set_index("instrument")["score_rank"]
    record("deterministic_ties",a.sort_index().equals(b.sort_index()))
    rejects("future_labels_rejected",lambda:rank_signals(raw.assign(label=999.0),cfg))
    account=empty_account_snapshot("ols",at,cfg)
    before=copy.deepcopy(account)
    targets,_=build_target_weights(ranked,account,known,cfg)
    qty,_=estimate_target_shares(targets,account,known,None,cfg)
    record("bootstrap_up_to_k",len(targets)==50)
    record("cash_budget_and_board_units",qty.estimated_buy_cash.sum()<=950000.01 and (qty.quantity_delta%100).eq(0).all())
    record("account_not_mutated",account["cash_available"]==before["cash_available"] and account["positions"].equals(before["positions"]))
    flat=rank_signals(raw.assign(score_raw=1.0),cfg)
    t,_=build_target_weights(flat,account,known,cfg)
    record("constant_scores_no_new_buys",t.empty)
    terminal=rank_signals(raw.assign(candidate_eligible=False,earliest_execution_date=pd.NaT),cfg)
    t,_=build_target_weights(terminal,account,known,cfg)
    record("missing_next_day_no_orders",t.empty)
    # 满仓账户：5个退出意图不提供尚未收到的卖出资金。
    held=codes[20:70]
    pos=pd.DataFrame(dict(instrument=held,quantity=1900,sellable_quantity=1900,frozen_quantity=0,
                         valuation_price=10.0,price_as_of=at)).set_index("instrument")
    funded=empty_account_snapshot("ols",at,cfg)
    funded.update(positions=pos,cash_available=50000.0)
    t,_=build_target_weights(ranked,funded,known,cfg)
    q,_=estimate_target_shares(t,funded,known,None,cfg)
    record("buffer_max_five_exits",t.intent.eq("exit").sum()==5)
    record("unfilled_sells_do_not_fund_buys",q.quantity_delta.gt(0).sum()==0 and q.quantity_delta.lt(0).sum()==5)
    repeat=copy.deepcopy(funded)
    repeat.update(replacement_counter_date=day,discretionary_replacements_used=5)
    repeated,_=build_target_weights(ranked,repeat,known,cfg)
    record("same_day_replacement_counter",repeated.intent.eq("exit").sum()==0)
    # 不可卖旧仓保留股数、占用席位和预算。
    frozen=copy.deepcopy(funded)
    frozen["positions"].loc[codes[-1],["sellable_quantity","frozen_quantity"]]=[0,1900]
    t,_=build_target_weights(ranked,frozen,known,cfg)
    q,_=estimate_target_shares(t,frozen,known,None,cfg)
    row=q.set_index("instrument").loc[codes[-1]]
    record("t_plus_one_frozen_position_kept",row.target_quantity_estimate==1900 and row.quantity_delta==0)
    missing=copy.deepcopy(funded)
    reduced=ranked.loc[~ranked.instrument.eq(codes[-1])]
    t,_=build_target_weights(reduced,missing,known,cfg)
    record("held_missing_score_kept",t.set_index("instrument").loc[codes[-1],"current_quantity"]==1900)
    # 当真实账户已经有待成交买单，不能重复占满席位。
    pending=copy.deepcopy(account)
    pending["open_orders"]=pd.DataFrame(dict(instrument=codes[:50],status="open"))
    t,_=build_target_weights(ranked,pending,known,cfg)
    record("pending_buys_reserve_slots",t.empty)
    future=known.copy();future["price_as_of"]=at+pd.Timedelta(days=1)
    rejects("future_price_rejected",lambda:estimate_target_shares(targets,account,future,None,cfg))
    bad_account=copy.deepcopy(account);bad_account["model"]="mlp"
    rejects("model_account_mismatch_rejected",lambda:build_target_weights(ranked,bad_account,known,cfg))
    wrong_target=targets.copy();wrong_target.loc[wrong_target.index[0],"current_quantity"]=100
    rejects("stale_target_quantities_rejected",lambda:estimate_target_shares(wrong_target,account,known,None,cfg))
    mixed_target=targets.copy();mixed_target.loc[mixed_target.index[0],"model"]="mlp"
    rejects("mixed_model_targets_rejected",lambda:estimate_target_shares(mixed_target,account,known,None,cfg))
    no_time=known.copy();no_time["price_as_of"]=pd.NaT
    rejects("unknown_price_time_rejected",lambda:estimate_target_shares(targets,account,no_time,None,cfg))
    # 严格模式的规则未知不能继续使用研究模式的板块假设。
    strict=dict(cfg,status_mode="strict")
    strict_targets,_=build_target_weights(ranked,account,known,strict)
    record("strict_market_state_rechecked",strict_targets.empty)
    verified_known=known.assign(historical_status_verified=True,known_can_buy=True,known_can_sell=True)
    strict_targets,_=build_target_weights(ranked,account,verified_known,strict)
    q,_=estimate_target_shares(strict_targets,account,verified_known,None,strict)
    record("strict_unknown_rules_no_buy",q.quantity_delta.eq(0).all())
    rejects("changed_target_config_rejected",lambda:estimate_target_shares(targets,account,known,None,dict(cfg,top_k=40)))
    changed_snapshot=copy.deepcopy(account)
    changed_snapshot["open_orders"]=pd.DataFrame(dict(instrument=[codes[-1]],status="open"))
    rejects("changed_snapshot_rejected",lambda:estimate_target_shares(targets,changed_snapshot,known,None,cfg))
    record("main_and_star_order_units",legal_order_quantity(209,100,100)==200 and legal_order_quantity(209,200,1)==209)
    # 六模型融合不使用 raw score 的量纲。
    frames={name:raw.assign(model=name,score_raw=raw.score_raw*(i+1)).copy() for i,name in enumerate(EXPECTED_MODELS)}
    for frame in frames.values():frame["source_hash"]="fixture"
    fused,_=build_rank_ensemble_day(frames,cfg)
    record("ensemble_rank_scale_invariance",fused.loc[fused.actionable,"instrument"].tolist()==codes)
    rejects("ensemble_missing_member_rejected",lambda:build_rank_ensemble_day({k:v for k,v in frames.items() if k!="mlp"},cfg))
    # 小型真实文件导出：保留无标签行，失败时 latest 不变化。
    work_dir=Path(work_dir or Path.cwd()/"tmp/signal_checks").resolve()
    work_dir.mkdir(parents=True,exist_ok=True)
    with tempfile.TemporaryDirectory(prefix="case_",dir=work_dir) as directory:
        root=Path(directory).resolve()
        if root.parent!=work_dir:
            raise ValueError("测试临时目录超出指定工作区。")
        fixture=raw.copy().rename(columns={"signal_date":"datetime","score_raw":"score"})
        fixture=fixture[["datetime","instrument","model","score"]].assign(role="walk_forward_test",fold=0,label=np.nan)
        paths={}
        for name in EXPECTED_MODELS:
            path=root/f"{name}.parquet"
            fixture.assign(model=name).set_index(["datetime","instrument"]).to_parquet(path)
            paths[name]=path
        synthetic_source=dict(models=EXPECTED_MODELS.copy(),rolling_run=root/"rolling",base_run=root/"base",
            row_counts={name:len(raw) for name in EXPECTED_MODELS},prediction_paths=paths,source_hash="fixture",
            windows=pd.DataFrame([dict(fold=0,predict_start=day,predict_end=day,predict_rows=len(raw))]),
            fold_headers={(0,name):dict(available_after="2024-12-31",max_train_label_end="2024-12-31") for name in EXPECTED_MODELS},
            verified_hashes=False,manifest={"test_fixture":True})
        daily=known.drop(columns=["as_of","price_as_of"]).reset_index().assign(signal_date=day,pre_close=10.0,
            volume_shares=100000.0,amount_yuan=1000000.0).set_index(["signal_date","instrument"])
        synthetic_market=dict(daily=daily,calendar=pd.DatetimeIndex([day,pd.Timestamp("2025-01-03")]),
            listing=pd.DataFrame(dict(instrument=codes,list_date=pd.Timestamp("2000-01-01"))).set_index("instrument"),
            source_files=[],assumptions=["test_fixture"])
        export_cfg=dict(cfg,timezone="Asia/Shanghai",signal_delay_minutes=30)
        normalized=normalize_model_signals(synthetic_source,"ols",synthetic_market,export_cfg)
        record("unlabeled_rows_retained",len(normalized)==70 and not FORBIDDEN_DECISION_FIELDS.intersection(normalized.columns))
        strict_norm=normalize_model_signals(synthetic_source,"ols",synthetic_market,dict(export_cfg,status_mode="strict"))
        record("strict_unknown_status_no_candidate",not strict_norm.candidate_eligible.any())
        published=export_signal_dataset(synthetic_source,synthetic_market,export_cfg,root/"out")
        reloaded=load_signal_data(published,model="ols",start=day,end=day,verify_hashes=True)
        record("partition_save_load_roundtrip",len(reloaded)==70)
        pointer=(root/"out/latest_run.json").read_bytes()
        malformed=fixture.assign(model="wrong").set_index(["datetime","instrument"])
        malformed.to_parquet(paths["ols"])
        rejects("failed_export_rejected",lambda:export_signal_dataset(synthetic_source,synthetic_market,export_cfg,root/"out"))
        record("failed_export_preserves_latest",(root/"out/latest_run.json").read_bytes()==pointer)
    return pd.DataFrame(records)


In [18]:
RUN_ACCEPTANCE_CHECKS = True
if RUN_ACCEPTANCE_CHECKS:
    acceptance_trace = text_io.StringIO()
    try:
        with contextlib.redirect_stdout(acceptance_trace):
            acceptance_report = run_signal_acceptance_checks(PROJECT_DIR / "tmp/signal_checks")
    except Exception:
        print(acceptance_trace.getvalue())
        raise
    display(acceptance_report)
    print(f"全部 {len(acceptance_report)} 项边界检查通过。")


,check,status
0,deterministic_ties,pass
1,future_labels_rejected,pass
2,bootstrap_up_to_k,pass
3,cash_budget_and_board_units,pass
4,account_not_mutated,pass
5,constant_scores_no_new_buys,pass
6,missing_next_day_no_orders,pass
7,buffer_max_five_exits,pass
8,unfilled_sells_do_not_fund_buys,pass
9,same_day_replacement_counter,pass


全部 30 项边界检查通过。


## 13. 后续回测内核怎样调用

真实事件循环在每个信号可用时点读取相应模型信号，取当前账户和已知市场快照，再调用本 notebook 的两个组合函数：

```python
day_signals = load_signal_data(signal_run, model, start=signal_date, end=signal_date)
targets, decisions = build_target_weights(day_signals, real_account, known_market, config)
intents, checks = estimate_target_shares(targets, real_account, known_prices, historical_rules, config)
# 接下来由 OrderPlanner、ExecutionModel、AccountLedger 处理订单、成交与账户。
```

真实引擎应依次处理公司行动与可卖数量解锁、预先存在订单的成交、账户估值、收盘后新信号。六模型使用独立账户；季度更换模型文件时，原账户持仓保持连续。未成交卖单不提供现金，缺分数或退市股票不会从账户中凭空消失。

待确认的替换买入应由真实成交反馈和新的账户快照解锁；未完成订单以及当日已用轮换计数必须进入下一次调用。

当前 `SIGNAL_RUN` 提供可复用的排名和资格数据，`PREVIEW_DIR` 提供明示假设的一日组合示例。历史 ST、停复牌、公司行动和完整费用规则补齐后，再验证执行约束版回测。

2025 年已经有静态测试与滚动预测评价。若之后依据这些结果调整新策略，新的未见期验证应重新登记。

## 14. 本 notebook 的输出与参考

你运行后应看到：

- 六模型来源与四个窗口表，所有来源校验通过。
- `data/processed/signals/run_.../`：分模型月份的信号、排名、配置、来源与质量报告。
- `reports/calendar_gaps.csv`：日历末端仍保留、但暂无下一执行日的记录。
- `previews/preview_.../`：六模型独立空仓快照的目标和股数意图。
- 自动验收表。原始预测文件、模型文件及其他 notebook 都作为只读输入。

策略与尺度参考：[Qlib 官方策略文档](https://qlib.readthedocs.io/en/stable/component/strategy.html)。板块数量差异参考：[上交所科创板规则投教](https://edu.sse.com.cn/tib/qa/c/4866268.shtml)。本 notebook 的缓冲规则按自己的配置和实现版本记录，不宣称与 Qlib 默认逐笔等价。

开发规范见项目中的 `signal.md`，整体规划见 `bactest_benchmark.md`。